# 02 - Data Pipeline (Module 1 final stage)

End-to-end, leakage-free data pipeline driven by a YAML config:

    raw CSV
    -> load_ett_dataset()
    -> time_based_split()
    -> prepare_scaled_splits()      (fit scaler on train only)
    -> prepare_windowed_dataloaders()  (windows built inside each split)
    -> train_loader / val_loader / test_loader

Requires PyTorch. Install dependencies first: `pip install -r ../requirements.txt`.

## 1. Imports

In [ ]:
import os
import sys

import numpy as np

PROJECT_ROOT = ".."
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.utils.config import load_config, print_config
from src.utils.seed import set_seed
from src.utils.device import print_device_info

from src.data.loader import load_ett_dataset
from src.data.splitter import time_based_split, print_split_summary
from src.data.preprocessing import (
    prepare_scaled_splits,
    print_preprocessing_summary,
)
from src.data.dataset import (
    prepare_windowed_dataloaders,
    print_window_summary,
)

## 2. Load configuration

Swap this path to `ETTh1_univariate_h24.yaml` to run the univariate setting.

In [ ]:
config = load_config("../configs/ETTh1_multivariate_h24.yaml")
print_config(config)

## 3. Set seed and device

In [ ]:
set_seed(config["training"]["seed"])
print_device_info()

## 4. Load dataset

In [ ]:
df = load_ett_dataset("../" + config["dataset"]["path"])
df.head()

## 5. Chronological split

In [ ]:
train_df, val_df, test_df = time_based_split(
    df=df,
    train_ratio=config["split"]["train_ratio"],
    val_ratio=config["split"]["val_ratio"],
    test_ratio=config["split"]["test_ratio"],
)
print_split_summary(train_df, val_df, test_df)

## 6. Fit scalers on train only, then transform all splits

In [ ]:
scaled_data = prepare_scaled_splits(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    input_type=config["dataset"]["input_type"],
    target_col=config["dataset"]["target"],
)
print_preprocessing_summary(scaled_data)

## 7. Build sliding windows inside each split + DataLoaders

In [ ]:
windowed_data = prepare_windowed_dataloaders(
    scaled_data=scaled_data,
    input_len=config["window"]["input_len"],
    horizon=config["window"]["horizon"],
    batch_size=config["training"]["batch_size"],
)
print_window_summary(windowed_data)

## 8. Inspect one batch

In [ ]:
batch_X, batch_y = next(iter(windowed_data["train_loader"]))
print("Batch X shape:", batch_X.shape)
print("Batch y shape:", batch_y.shape)

## 9. Verify boundary handling (cross-split windows dropped)

In [ ]:
input_len = config["window"]["input_len"]
horizon = config["window"]["horizon"]

expected_train = len(train_df) - input_len - horizon + 1
expected_val = len(val_df) - input_len - horizon + 1
expected_test = len(test_df) - input_len - horizon + 1

print("Expected / actual train samples:", expected_train, windowed_data["train_X"].shape[0])
print("Expected / actual val samples:  ", expected_val, windowed_data["val_X"].shape[0])
print("Expected / actual test samples: ", expected_test, windowed_data["test_X"].shape[0])

## 10. (Optional) Save processed windows

Saved under `data/processed/`; ignored by git, kept locally only.

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

out_name = f"{config['dataset']['name']}_{config['dataset']['input_type']}_h{horizon}_windows.npz"
np.savez_compressed(
    f"../data/processed/{out_name}",
    train_X=windowed_data["train_X"],
    train_y=windowed_data["train_y"],
    val_X=windowed_data["val_X"],
    val_y=windowed_data["val_y"],
    test_X=windowed_data["test_X"],
    test_y=windowed_data["test_y"],
)
print("Saved:", out_name)

## 11. Module 1 closed

The pipeline now follows the required leakage-free order: split -> fit scaler on
train only -> transform all splits -> build windows inside each split ->
DataLoaders. Downstream Modules 2 and 3 consume `train_loader / val_loader /
test_loader` and `num_features` directly from `windowed_data`.